# Práctica guiada: ISOMAP y comparación de métodos de reducción de dimensionalidad

Entorno recomendado: Google Colab  
Dataset: Wine de `scikit-learn`

## Resultados de aprendizaje

1. Estandarizar variables antes de aplicar métodos basados en distancias.
2. Aplicar ISOMAP para obtener una representación bidimensional.
3. Analizar el efecto del número de vecinos sobre la representación obtenida con ISOMAP.
4. Comparar PCA, ICA, LLE e ISOMAP utilizando una métrica común de preservación local.
5. Analizar si una mejor preservación de vecindarios implica necesariamente un mejor desempeño en una tarea de clasificación.
6. Interpretar las diferencias entre métodos lineales y no lineales de reducción de dimensionalidad.


## Instrucciones

- Ejecute las celdas en orden.
- Complete únicamente las líneas de código indicadas.
- Responda las preguntas incluidas en las celdas Markdown.
- No cambie los nombres de las variables.
- Ejecute la prueba `assert` después de cada actividad.
- Las etiquetas de clase se utilizarán para visualizar y evaluar las representaciones, pero no para ajustar los métodos de reducción de dimensionalidad.
- Para que la comparación sea justa, todos los métodos recibirán los mismos datos estandarizados y producirán dos componentes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from sklearn.manifold import Isomap, LocallyLinearEmbedding, trustworthiness
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score


## 1. Carga y exploración de los datos

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data.copy()
y = wine.target.copy()

print("Dimensiones de X:", X.shape)
print("Número de clases:", y.nunique())
display(X.head())


In [ ]:
resumen = pd.DataFrame({
    "media": X.mean(),
    "desviación estándar": X.std(),
    "mínimo": X.min(),
    "máximo": X.max()
})

display(resumen.round(3))


Pregunta 1. Observe el resumen de las variables.

1. ¿Las variables se encuentran en escalas similares?
2. ¿Por qué las diferencias de escala son especialmente importantes para ISOMAP?
3. ¿En qué etapa de ISOMAP se utilizan directamente las distancias entre observaciones?

Respuesta:


## 2. Estandarización

Como ISOMAP, LLE y otros métodos utilizan relaciones geométricas entre observaciones, se estandarizarán las variables antes de construir las representaciones.


In [ ]:
# ACTIVIDAD 1
# Cree una instancia de StandardScaler y aplíquela sobre X.

# scaler =
# X_scaled =

print("Forma de X_scaled:", X_scaled.shape)


In [ ]:
# PRUEBA 1
assert X_scaled.shape == X.shape
assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-10)
assert np.allclose(X_scaled.std(axis=0), 1, atol=1e-10)
print("✓ Datos estandarizados correctamente")


## 3. Aplicación de ISOMAP con dos componentes

En esta primera aplicación se utilizarán 10 vecinos y se reducirá el conjunto de datos a dos dimensiones.

ISOMAP construye un grafo de vecindad, estima las distancias geodésicas mediante caminos mínimos y aplica MDS clásico para obtener las nuevas coordenadas.


In [ ]:
# ACTIVIDAD 2
# Cree un modelo ISOMAP con:
# n_neighbors=10
# n_components=2

# isomap_2d =

# Aplique el modelo sobre los datos estandarizados.
# X_isomap =

print("Forma de X_isomap:", X_isomap.shape)


In [ ]:
# PRUEBA 2
assert isinstance(isomap_2d, Isomap)
assert isomap_2d.n_neighbors == 10
assert isomap_2d.n_components == 2
assert X_isomap.shape == (X.shape[0], 2)
assert np.isfinite(X_isomap).all()
print("✓ ISOMAP produjo una representación bidimensional")


Pregunta 2. Explique el papel de `n_neighbors` en ISOMAP.

1. ¿Qué puede ocurrir si el valor es demasiado pequeño?
2. ¿Qué puede ocurrir si es demasiado grande?
3. ¿Cómo puede afectar este parámetro la estimación de las distancias geodésicas?

Respuesta:


## 4. Visualización de la representación obtenida con ISOMAP

In [ ]:
# ACTIVIDAD 3
# Genere un diagrama de dispersión usando las dos coordenadas de X_isomap.
# Utilice y únicamente para asignar color a los puntos.

plt.figure(figsize=(8, 6))

# scatter = plt.scatter(
#     ,
#     ,
#     c=y,
#     alpha=0.75
# )

plt.xlabel("Coordenada ISOMAP 1")
plt.ylabel("Coordenada ISOMAP 2")
plt.title("Representación ISOMAP del dataset Wine")
plt.grid(alpha=0.25)
plt.colorbar(scatter, label="Clase usada solo para visualización")
plt.show()


In [ ]:
# PRUEBA 3
assert X_isomap[:, 0].shape[0] == len(y)
assert X_isomap[:, 1].shape[0] == len(y)
assert not np.allclose(X_isomap[:, 0], X_isomap[:, 1])
print("✓ Coordenadas compatibles con la visualización")


Pregunta 3. Observe la representación obtenida.

1. ¿Se identifican agrupaciones o regiones diferenciadas?
2. ¿Qué clases presentan mayor superposición?
3. ¿La separación observada implica que ISOMAP utilizó las etiquetas?
4. ¿Qué significa aproximadamente que dos puntos aparezcan cercanos en el espacio reducido?

Respuesta:


## 5. Efecto del número de vecinos en ISOMAP

Se entrenarán varios modelos cambiando solamente `n_neighbors`. Todos producirán dos componentes.


In [ ]:
# ACTIVIDAD 4
# Entrene un modelo ISOMAP para cada valor de vecinos
# y almacene cada embedding en resultados_isomap.

valores_vecinos = [5, 10, 15, 30]
resultados_isomap = {}

# for k in valores_vecinos:
#     modelo =
#     embedding =
#     resultados_isomap[k] = embedding


In [ ]:
# PRUEBA 4
assert set(resultados_isomap.keys()) == set(valores_vecinos)

for k in valores_vecinos:
    assert resultados_isomap[k].shape == (X.shape[0], 2)
    assert np.isfinite(resultados_isomap[k]).all()

print("✓ Todos los modelos ISOMAP fueron entrenados")


In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(13, 10))

for eje, k in zip(ejes.ravel(), valores_vecinos):
    embedding = resultados_isomap[k]

    scatter = eje.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=y,
        alpha=0.75
    )

    eje.set_title(f"n_neighbors = {k}")
    eje.set_xlabel("Coordenada ISOMAP 1")
    eje.set_ylabel("Coordenada ISOMAP 2")
    eje.grid(alpha=0.25)

fig.colorbar(
    scatter,
    ax=ejes.ravel().tolist(),
    label="Clase usada solo para visualización"
)

fig.suptitle("Efecto del número de vecinos sobre ISOMAP", fontsize=14)
plt.show()


Pregunta 4. Compare las cuatro representaciones.

1. ¿La geometría del embedding cambia al variar `n_neighbors`?
2. ¿Qué valores parecen producir una representación más estable?
3. ¿Observa indicios de pérdida de estructura cuando el vecindario aumenta?
4. ¿Cuál configuración escogería solamente a partir de la inspección visual? Justifique.

Respuesta:


## 6. Evaluación de ISOMAP mediante trustworthiness

`trustworthiness` mide hasta qué punto los vecinos que aparecen próximos en el espacio reducido también eran cercanos en el espacio original.

Para comparar de manera controlada todas las configuraciones se utilizará el mismo tamaño de vecindario de evaluación: `k_eval = 10`.


In [ ]:
# ACTIVIDAD 5
# Calcule trustworthiness para cada embedding ISOMAP usando k_eval=10.

k_eval = 10
# trust_isomap = {}

# for k in valores_vecinos:
#     valor =
#     trust_isomap[k] = valor

print(trust_isomap)


In [ ]:
# PRUEBA 5
assert set(trust_isomap.keys()) == set(valores_vecinos)

for valor in trust_isomap.values():
    assert 0 <= valor <= 1

print("✓ Trustworthiness calculado para todas las configuraciones")


In [ ]:
valores_trust_isomap = [trust_isomap[k] for k in valores_vecinos]

plt.figure(figsize=(8, 5))
barras = plt.bar(valores_vecinos, valores_trust_isomap)

for barra, valor in zip(barras, valores_trust_isomap):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.01,
        f"{valor:.4f}",
        ha="center",
        va="bottom"
    )

plt.xlabel("n_neighbors usado por ISOMAP")
plt.ylabel(f"Trustworthiness evaluado con k = {k_eval}")
plt.title("Preservación local para diferentes valores de n_neighbors")
plt.ylim(0, 1)
plt.grid(True, axis="y", alpha=0.25)
plt.show()


Pregunta 5. Analice los resultados de `trustworthiness`.

1. ¿Qué valor de `n_neighbors` obtiene la puntuación más alta?
2. ¿Coincide con la configuración que había elegido visualmente?
3. ¿Una diferencia pequeña en `trustworthiness` necesariamente implica una diferencia importante en la utilidad del embedding?
4. ¿Por qué se utiliza el mismo `k_eval` para evaluar todas las configuraciones?

Respuesta:


## 7. Comparación entre PCA, ICA, LLE e ISOMAP

Ahora se compararán cuatro métodos estudiados durante la unidad:

- PCA: método lineal que busca direcciones de máxima varianza.
- ICA: método lineal que busca componentes estadísticamente independientes.
- LLE: método no lineal que intenta preservar relaciones locales de reconstrucción.
- ISOMAP: método no lineal que intenta preservar distancias geodésicas globales.

Todos los métodos utilizarán los mismos datos estandarizados y producirán dos componentes.

Para LLE e ISOMAP se utilizarán 10 vecinos con el fin de mantener una comparación controlada.


In [ ]:
# ACTIVIDAD 6
# Cree y ajuste los cuatro métodos con dos componentes.
# Para LLE e ISOMAP utilice n_neighbors=10.

# pca =
# ica =
# lle =
# isomap =

# X_pca =
# X_ica =
# X_lle =
# X_isomap_comp =


In [ ]:
# PRUEBA 6
embeddings = {
    "PCA": X_pca,
    "ICA": X_ica,
    "LLE": X_lle,
    "ISOMAP": X_isomap_comp
}

for nombre, embedding in embeddings.items():
    assert embedding.shape == (X.shape[0], 2)
    assert np.isfinite(embedding).all()

print("✓ Los cuatro métodos produjeron representaciones bidimensionales")


In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(13, 10))

for eje, (nombre, embedding) in zip(ejes.ravel(), embeddings.items()):
    scatter = eje.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=y,
        alpha=0.75
    )
    eje.set_title(nombre)
    eje.set_xlabel("Componente 1")
    eje.set_ylabel("Componente 2")
    eje.grid(alpha=0.25)

fig.colorbar(
    scatter,
    ax=ejes.ravel().tolist(),
    label="Clase usada solo para visualización"
)

fig.suptitle("Comparación de representaciones bidimensionales", fontsize=14)
plt.show()


Pregunta 6. Compare visualmente los cuatro métodos.

1. ¿Cuál produce la separación visual más clara entre clases?
2. ¿Cuál muestra mayor superposición?
3. ¿Las formas geométricas producidas por los cuatro métodos son similares?
4. ¿Por qué no debería concluirse que un método es mejor únicamente porque separa visualmente mejor las clases?
5. Relacione las diferencias observadas con el objetivo matemático de cada método.

Respuesta:


## 8. Comparación cuantitativa con una métrica común

No es apropiado comparar directamente métricas internas específicas de métodos diferentes, porque cada algoritmo optimiza un criterio distinto.

Para realizar una comparación común se calculará `trustworthiness` sobre las cuatro representaciones utilizando `k_eval = 10`.


In [ ]:
# ACTIVIDAD 7
# Calcule trustworthiness para cada método.

k_eval = 10
# trust_metodos = {}

# for nombre, embedding in embeddings.items():
#     trust_metodos[nombre] =

comparacion_trust = pd.DataFrame({
    "Método": list(trust_metodos.keys()),
    "Trustworthiness": list(trust_metodos.values())
}).sort_values("Trustworthiness", ascending=False)

display(comparacion_trust.round(4))


In [ ]:
# PRUEBA 7
assert set(trust_metodos.keys()) == {"PCA", "ICA", "LLE", "ISOMAP"}
for valor in trust_metodos.values():
    assert 0 <= valor <= 1
print("✓ Comparación de trustworthiness completada")


In [ ]:
plt.figure(figsize=(8, 5))
barras = plt.bar(comparacion_trust["Método"], comparacion_trust["Trustworthiness"])

for barra, valor in zip(barras, comparacion_trust["Trustworthiness"]):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.01,
        f"{valor:.4f}",
        ha="center",
        va="bottom"
    )

plt.ylabel(f"Trustworthiness evaluado con k = {k_eval}")
plt.title("Preservación local de los cuatro métodos")
plt.ylim(0, 1)
plt.grid(True, axis="y", alpha=0.25)
plt.show()


Pregunta 7. Analice la comparación mediante `trustworthiness`.

1. ¿Qué método obtuvo el valor más alto?
2. ¿Coincide con el método que parecía mejor en la inspección visual?
3. ¿Por qué PCA o ICA podrían obtener un valor competitivo aunque no sean métodos de aprendizaje de variedades?
4. ¿Puede utilizarse `trustworthiness` como criterio absoluto para afirmar que un método es superior a todos los demás? Explique.

Respuesta:


## 9. Comparación en una tarea posterior: clasificación

Una representación reducida también puede evaluarse por su utilidad en una tarea posterior.

Se utilizará regresión logística multiclase y validación cruzada estratificada de 5 particiones. La misma estrategia de validación se aplicará a las cuatro representaciones.

Esta comparación no convierte los métodos de reducción en supervisados: las etiquetas intervienen únicamente después de obtener los embeddings, durante la evaluación del clasificador.


In [ ]:
# ACTIVIDAD 8
# Evalúe una regresión logística sobre cada embedding.
# Use StratifiedKFold con 5 particiones, shuffle=True y random_state=42.
# Calcule el accuracy promedio mediante cross_val_score.

# cv =
# accuracy_metodos = {}

# for nombre, embedding in embeddings.items():
#     clasificador =
#     scores =
#     accuracy_metodos[nombre] =

comparacion_final = pd.DataFrame({
    "Método": list(embeddings.keys()),
    "Trustworthiness": [trust_metodos[n] for n in embeddings.keys()],
    "Accuracy_CV": [accuracy_metodos[n] for n in embeddings.keys()]
})

display(comparacion_final.round(4))


In [ ]:
# PRUEBA 8
assert set(accuracy_metodos.keys()) == {"PCA", "ICA", "LLE", "ISOMAP"}
for valor in accuracy_metodos.values():
    assert 0 <= valor <= 1
print("✓ Accuracy calculado para las cuatro representaciones")


Pregunta 8. Compare `trustworthiness` y accuracy.

1. ¿El método con mayor `trustworthiness` también obtiene el mayor accuracy?
2. ¿Qué significa si los dos criterios seleccionan métodos diferentes?
3. ¿Qué propiedad parece ser más importante para la regresión logística en este conjunto de datos?
4. ¿Puede concluirse que una representación con mejor separación visual siempre es mejor para clasificación?
5. Si el objetivo fuera únicamente visualización, ¿usaría necesariamente el mismo método que para clasificación? Justifique.

Respuesta:


## 10. Síntesis

Complete la siguiente tabla conceptualmente a partir de los resultados de la práctica.

| Método | Lineal / no lineal | Propiedad principal que busca preservar o extraer | Resultado observado |
|---|---|---|---|
| PCA |  |  |  |
| ICA |  |  |  |
| LLE |  |  |  |
| ISOMAP |  |  |  |

Pregunta 9. Redacte una conclusión de entre 8 y 12 líneas en la que responda:

1. ¿Qué diferencias fundamentales observó entre los métodos lineales y no lineales?
2. ¿Qué método preservó mejor los vecindarios?
3. ¿Qué método produjo la representación más útil para clasificación?
4. ¿Qué papel tuvo `n_neighbors` en ISOMAP?
5. ¿Por qué no existe necesariamente un único método de reducción de dimensionalidad que sea mejor para todos los objetivos?

Respuesta:


# Práctica adicional

Diseñe un pequeño experimento para investigar si aumentar el número de componentes cambia la conclusión de la comparación.

1. Pruebe `n_components = [2, 3, 5]`.
2. Aplique PCA, ICA, LLE e ISOMAP para cada número de componentes.
3. Calcule el accuracy de regresión logística con validación cruzada.
4. Compare los resultados.
5. Determine si el método que parece mejor con dos componentes continúa siendo el mejor cuando se conserva más información.

No utilice las etiquetas para ajustar los métodos de reducción de dimensionalidad.


In [ ]:
# ESPACIO PARA LA PRÁCTICA ADICIONAL

